In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [14]:
class ScaledDotProductAttention(nn.Module):
    '''
    Scaled Dot Product Attention
    attention(Q,K,V) = softmax((Q*K^T)/sqrt(d_k))*V
    '''
    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()
        
    def forward(self, query, key, value, mask=None):
        '''
        
        :param query: 查询向量，[batch_size, num_heads, seq_len, d_model] 
        :param key:  键值向量， [batch_size, num_heads, seq_len, d_model]
        :param value: 值向量， [batch_size, num_heads, seq_len, d_model]
        :param mask [[1,0,0],[1,1,0], [1,1,1]]
        :return: 
        '''
        d_k= key.size()[-1]
        # scores [batch_size, seq_len, seq_len]
        scores = torch.matmul(query, key.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32)) 
        
        if mask is not None:
            scores = scores.masked_fill(mask==0, -1e9)
        scores = torch.softmax(scores, dim=-1)
        # output [batch_size, seq_len, d_model]  broadcast
        output = torch.matmul(scores, value)
        
        return output, scores

In [15]:
sdpa = ScaledDotProductAttention()

batch_size, seq_len, d_model = 16, 10, 768

query = torch.randn(batch_size, seq_len, d_model)
key = torch.randn(batch_size, seq_len, d_model)
value = torch.randn(batch_size, seq_len, d_model)

output, scores = sdpa(query, key, value)

print(f"output size is {output.size()}")
print(f"scores size is {scores.size()}")
print(f"score size is {scores}")


output size is torch.Size([16, 10, 768])
scores size is torch.Size([16, 10, 10])
score size is tensor([[[0.0183, 0.0940, 0.1047,  ..., 0.0297, 0.0243, 0.0890],
         [0.0251, 0.1126, 0.0461,  ..., 0.1860, 0.1333, 0.0934],
         [0.1621, 0.1220, 0.0615,  ..., 0.0450, 0.0462, 0.0359],
         ...,
         [0.0465, 0.1952, 0.0108,  ..., 0.0916, 0.0844, 0.0903],
         [0.1229, 0.0890, 0.0982,  ..., 0.0151, 0.1437, 0.0958],
         [0.0731, 0.0603, 0.1492,  ..., 0.0193, 0.0171, 0.0674]],

        [[0.0968, 0.2339, 0.0393,  ..., 0.0622, 0.0433, 0.0367],
         [0.0556, 0.1733, 0.0897,  ..., 0.0604, 0.0492, 0.4246],
         [0.1716, 0.1325, 0.0169,  ..., 0.0753, 0.1968, 0.0451],
         ...,
         [0.1390, 0.0549, 0.0617,  ..., 0.1536, 0.1940, 0.2411],
         [0.1232, 0.0731, 0.0929,  ..., 0.0646, 0.0581, 0.0886],
         [0.1625, 0.1820, 0.0865,  ..., 0.0549, 0.0482, 0.0600]],

        [[0.0761, 0.0606, 0.3837,  ..., 0.0217, 0.0294, 0.0687],
         [0.1638, 0.0763, 0.

In [16]:
sum(scores[0][0])

tensor(1.)

In [17]:
mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)
mask.size()

torch.Size([1, 10, 10])

In [18]:
output, scores = sdpa(query, key, value, mask)

print(f"output size is {output.size()}")
print(f"scores size is {scores.size()}")
print(f"score size is {scores}")

output size is torch.Size([16, 10, 768])
scores size is torch.Size([16, 10, 10])
score size is tensor([[[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.1821, 0.8179, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.4691, 0.3530, 0.1780,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.0564, 0.2365, 0.0131,  ..., 0.1110, 0.0000, 0.0000],
         [0.1360, 0.0985, 0.1086,  ..., 0.0166, 0.1589, 0.0000],
         [0.0731, 0.0603, 0.1492,  ..., 0.0193, 0.0171, 0.0674]],

        [[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.2430, 0.7570, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.5346, 0.4128, 0.0525,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.2462, 0.0972, 0.1092,  ..., 0.2719, 0.0000, 0.0000],
         [0.1351, 0.0802, 0.1019,  ..., 0.0708, 0.0638, 0.0000],
         [0.1625, 0.1820, 0.0865,  ..., 0.0549, 0.0482, 0.0600]],

        [[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.6822, 0.3178, 0.

In [19]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        self.d_model = d_model
        self.num_heads = num_heads
        
        super(MultiHeadAttention, self).__init__()
        assert self.d_model%self.num_heads==0, "d_model must be divisible by num_heads"
        
        self.head_dim = self.d_model//self.num_heads
        
        self.query_proj = nn.Linear(d_model, d_model)
        self.key_proj = nn.Linear(d_model, d_model)
        self.value_proj = nn.Linear(d_model, d_model)
        
        self.out_proj = nn.Linear(d_model, d_model)
        self.attention = ScaledDotProductAttention()
    
    def split_heads(self, x):
        '''
        
        :param x: [batch_size, seq_len, d_model]
        :return: 
            [batch_size, num_heads, seq_len, head_dim]
        '''
        batch_size, seq_len, d_model = x.size()
        assert  d_model==self.head_dim*self.num_heads, f"input must in dim {self.num_heads*self.head_dim} but input dim is {d_model}"
        
        return x.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1,2)
    def combine_heads(self,x):
        '''
        
        :param x:  [batch_size, num_heads, seq_len, head_dim]
        :return:  [batch_size, seq_len, num_heads*head_dim]
        '''
        batch_size, num_heads, seq_len, head_dim = x.size()
        
        return x.transpose(1,2).contiguous().view(batch_size, seq_len,num_heads*head_dim)
        
    def forward(self, x, mask=None):
        
        query = self.query_proj(x)
        key = self.key_proj(x)
        value = self.value_proj(x)
        
        splited_query = self.split_heads(query)
        splited_key = self.split_heads(key)
        splited_value = self.split_heads(value)
        
        output, scores =self.attention(splited_query, splited_key, splited_value, mask) 
        output = self.combine_heads(output)
        
        return self.out_proj(output), scores
        
        
        

In [20]:
class MaskedMultiheadAttention(MultiHeadAttention):
    def __init__(self, d_model, num_heads):
        super(MaskedMultiheadAttention, self).__init__(d_model, num_heads)
        
    def forward(self, x):
        seq_len = x.size()[1]
        mask = torch.tril(torch.ones(1, seq_len,seq_len))
        return super().forward(x, mask)
        

In [21]:
batch_size, seq_len, d_model = 16, 10, 768
x = torch.randn(batch_size, seq_len, d_model)

mmha = MaskedMultiheadAttention(d_model, num_heads=12)
output, socres = mmha(x)

print(f"output size is {output.size()}")
print(f"scores size is {scores.size()}")
print(f"score size is {scores}")

output size is torch.Size([16, 10, 768])
scores size is torch.Size([16, 10, 10])
score size is tensor([[[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.1821, 0.8179, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.4691, 0.3530, 0.1780,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.0564, 0.2365, 0.0131,  ..., 0.1110, 0.0000, 0.0000],
         [0.1360, 0.0985, 0.1086,  ..., 0.0166, 0.1589, 0.0000],
         [0.0731, 0.0603, 0.1492,  ..., 0.0193, 0.0171, 0.0674]],

        [[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.2430, 0.7570, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.5346, 0.4128, 0.0525,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.2462, 0.0972, 0.1092,  ..., 0.2719, 0.0000, 0.0000],
         [0.1351, 0.0802, 0.1019,  ..., 0.0708, 0.0638, 0.0000],
         [0.1625, 0.1820, 0.0865,  ..., 0.0549, 0.0482, 0.0600]],

        [[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.6822, 0.3178, 0.

In [22]:
# Transfomer Decoder:
# 1. token embedding
# 2. position embedding
# 3. Multihead attention
# 4. Feed forward Nural network
# 5. attention +FFN ：decoder block
# 6. 多个decoder堆叠起来
# 7.laynorm、activation、残差连接

In [23]:
import torch
import torch.nn as nn

class FeedForwardNeuralNetwork(nn.Module):
    def __init__(self, d_model, d_ff):
        super(FeedForwardNeuralNetwork, self).__init__()
        # laynorm
        self.layer_norm = nn.LayerNorm(d_model)
        # proj
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        #激活函数
        self.activation = nn.GELU()
        
    def forward(self, x):
        '''
        
        :param x: [batch_szie, seq_len, hidden_size]
        :return: 
        '''
        resiual = x
        output = self.layer_norm(x) #[batch_szie, seq_len, hidden_size]
        output = self.linear1(output) #[batch_szie, seq_len, hidden_size*4]
        output = self.activation(output)#[batch_szie, seq_len, hidden_size*4]
        output = self.linear2(output)#[batch_szie, seq_len, hidden_size]
        
        return resiual + output
        
        

In [24]:
batch_size, seq_len, hidden_size = 16, 10, 768

x = torch.randn(batch_size, seq_len, hidden_size)

ffn = FeedForwardNeuralNetwork(768, 768*4)

output = ffn(x)

print(f"x size is {x.size()}")
print(f"output size is {output.size()}")
print(f"output is {output[0]}")

x size is torch.Size([16, 10, 768])
output size is torch.Size([16, 10, 768])
output is tensor([[ 2.1975, -0.2907,  0.8418,  ...,  0.2823,  1.5420,  0.4670],
        [-1.6087, -2.0050,  0.4378,  ...,  0.5885, -2.0200,  0.4276],
        [ 0.5089,  1.5230,  0.2901,  ...,  0.0582, -0.4557,  0.9523],
        ...,
        [-0.3801,  0.6167, -0.1569,  ...,  0.8266,  0.0266,  0.7384],
        [ 0.8240,  0.3957,  0.3141,  ...,  0.9997,  1.1223, -2.3958],
        [-0.2163, -1.0351,  0.5196,  ...,  1.1651, -1.2975,  0.9898]],
       grad_fn=<SelectBackward0>)


In [25]:
class TransformerDecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super(TransformerDecoderBlock, self).__init__()
        #核心模块
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.feedForward = FeedForwardNeuralNetwork(d_model, d_ff)
        # 后层归一化
        self.layer_norm = nn.LayerNorm(d_model)
        
    def forward(self, x, attn_mask = None):
        attn_output, attn_weights = self.attention(x, attn_mask)
        
        ff_output = self.feedForward(x+attn_output)
        
        output = self.layer_norm(ff_output)
        
        return output, attn_weights

In [27]:
batch_size, seq_len, hidden_size = 16, 10, 768

x = torch.randn(batch_size, seq_len, hidden_size)

tdb = TransformerDecoderBlock(hidden_size, 12, hidden_size*4)

output, attn_weights = tdb(x)
print(f"x size is {x.size()}")
print(f"output size is {output.size()}")
print(f"output is {output[0]}")

x size is torch.Size([16, 10, 768])
output size is torch.Size([16, 10, 768])
output is tensor([[ 0.7520,  0.3359,  0.1926,  ...,  1.1546,  1.5551,  0.4757],
        [ 0.0729, -0.5593, -0.5886,  ...,  2.0179,  0.7397,  0.6211],
        [-1.0018,  0.8655,  0.1288,  ..., -0.5117, -1.4394,  0.3744],
        ...,
        [ 0.5960,  0.4915,  1.0358,  ...,  1.7666,  0.3547,  1.4238],
        [-0.7104, -0.1157,  0.7006,  ..., -0.7298, -1.0657,  0.1663],
        [ 1.1701,  0.6185, -0.0307,  ...,  1.7182, -1.1482, -0.5306]],
       grad_fn=<SelectBackward0>)


In [28]:
import math

class PositionalEncoding(nn.Module):
    """位置编码模块（支持动态序列长度）"""
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x):
        # 动态获取位置编码
        position_emb = self.pe[:, :x.size(1)]
        return x + position_emb  # [batch, seq_len, d_model]

In [37]:
# Transformer实现

class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, max_len, num_layers, num_heads, d_ff):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len)
        
        #堆叠Decoder块
        self.layers = nn.ModuleList([
            TransformerDecoderBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.output_layer = nn.Linear(d_model, vocab_size, bias=False)
        
        # tied embeddings
        self.output_layer.weight = self.token_embedding.weight
        
        self.init_weights()
        pass
    def init_weights(self):
        nn.init.normal_(self.token_embedding.weight, std=0.02)
        
        #各层做一下初始化
        for layer in self.layers:
            nn.init.xavier_normal(layer.attention.query_proj.weight)
            nn.init.xavier_normal(layer.attention.key_proj.weight)
            nn.init.xavier_normal(layer.attention.value_proj.weight)
            
            nn.init.kaiming_normal(layer.feedForward.linear1.weight)
            nn.init.kaiming_uniform(layer.feedForward.linear2.weight)
    
    def create_causal_mask(self, seq_len):
        mask = torch.tril(torch.ones(seq_len,seq_len))
        return mask
    
    def forward(self, input_ids):
        '''
        
        :param x:[batch_size, seq_len]  
        :return: 
        '''
        batch_size, seq_len = input_ids.size()
        
        #嵌入
        embeddings = self.token_embedding(input_ids) #[batch_size, seq_len, d_model]
        pos_embedding = self.pos_encoder(embeddings)
        
        embeddings = embeddings + pos_embedding
        
        mask = self.create_causal_mask(seq_len)
        # 通过所有的Transformer Decoder block
        hidden_states = embeddings
        all_attn_weights = []
        for layer in self.layers:
            hidden_states, attn_weights = layer(hidden_states, mask)
            all_attn_weights.append(attn_weights)
        
        hidden_states = self.final_norm(hidden_states)
        
        logits = self.output_layer(hidden_states)
        
        return logits, all_attn_weights

In [38]:
model = TransformerDecoder(
    vocab_size=500, d_model=256, max_len=128, num_layers=12, num_heads=8, d_ff=256*4)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13180\1793193392.py:27: FutureWarning: `nn.init.xavier_normal` is now deprecated in favor of `nn.init.xavier_normal_`.
  nn.init.xavier_normal(layer.attention.query_proj.weight)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13180\1793193392.py:28: FutureWarning: `nn.init.xavier_normal` is now deprecated in favor of `nn.init.xavier_normal_`.
  nn.init.xavier_normal(layer.attention.key_proj.weight)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13180\1793193392.py:29: FutureWarning: `nn.init.xavier_normal` is now deprecated in favor of `nn.init.xavier_normal_`.
  nn.init.xavier_normal(layer.attention.value_proj.weight)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13180\1793193392.py:31: FutureWarning: `nn.init.kaiming_normal` is now deprecated in favor of `nn.init.kaiming_normal_`.
  nn.init.kaiming_normal(layer.feedForward.linear1.weight)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13180\1793193392.py:32: FutureWarning: `nn.init.kaiming_uniform

In [40]:
batch_size, seq_len = 16,10

input_ids = torch.randint(0,500,(batch_size, seq_len))

print(f"input size is {input_ids.size()}")
print(f"input is {input_ids}")

output, _ = model(input_ids)

print(f"output size is {output.size()}")
print(f"output is {output[-1]}")

input size is torch.Size([16, 10])
input is tensor([[179,  36, 296, 101,  28, 268, 269,  27,  39, 397],
        [408,  17, 186, 458, 339, 264, 249, 437, 286,  40],
        [ 92, 352, 341, 138,  69, 459, 113, 106, 394, 317],
        [  5,  45, 421, 366, 361,  49, 410, 212, 303,  19],
        [132, 420, 273, 114, 263, 446, 365, 442, 311, 367],
        [428, 180,  83, 378, 173, 108, 350, 447, 444, 247],
        [262,  37,  68,  33, 460, 183,  72,  57, 151, 218],
        [433,  76, 388,  81, 487,  33,  24, 325,  74, 406],
        [392, 323,  78, 207, 189,  82, 206, 240, 341, 334],
        [ 86, 303, 276, 127, 261,  17, 335, 417, 498, 119],
        [217, 112,  20, 426,  87, 202, 391, 396, 497, 178],
        [265, 327, 260, 495, 318,  22, 414,  77, 258, 262],
        [283, 395, 116,  75, 294, 284, 484, 163,  84, 471],
        [133, 182, 492, 395, 180, 478, 403, 459, 393, 257],
        [374, 129, 208, 447, 453, 106, 401, 107, 286, 487],
        [ 53, 377, 440, 282, 116, 324, 308, 380, 278, 26